# Partition Matrices and LU Decomposition

**Syllabus mapping:** partition matrix and their properties,
systems of linear equations and solutions, Gaussian elimination,
LU decomposition.

**Objectives:** understand block matrix multiplication, determinants
of block triangular matrices, and Schur complements; compute LU
decomposition $A = LU$ and solve linear systems via forward/back substitution.

## Theoretical Foundations

### 1. Partition (Block) Matrices
For conformably partitioned matrices:
$$\begin{bmatrix} A & B \\ C & D \end{bmatrix} \begin{bmatrix} X \\ Y \end{bmatrix} = \begin{bmatrix} AX + BY \\ CX + DY \end{bmatrix}$$

For block triangular matrices:
$$\det \begin{bmatrix} A & B \\ 0 & D \end{bmatrix} = \det(A) \det(D)$$

When $A$ is invertible, the block inverse is given via the Schur complement $S = D - CA^{-1}B$:
$$\det \begin{bmatrix} A & B \\ C & D \end{bmatrix} = \det(A) \det(D - CA^{-1}B)$$

### 2. LU Decomposition
Gaussian elimination without row interchanges factors a matrix $A$ into $A = LU$, where:
- $L$ is unit lower triangular ($l_{ii} = 1$, $l_{ij} = 0$ for $j > i$).
- $U$ is upper triangular ($u_{ij} = 0$ for $i > j$).

Solving $Ax = b$ becomes two $O(n^2)$ triangular solves:
1. Forward substitution: $Ly = b$
2. Back substitution: $Ux = y$

In [ ]:
import numpy as np

# 1. Partition Matrix Determinant & Multiplication
A = np.array([[2.0, 1.0], [1.0, 3.0]])
B = np.array([[1.0, 0.0], [2.0, 1.0]])
zero_block = np.zeros((2, 2))
D = np.array([[4.0, 2.0], [1.0, 2.0]])

# Construct 4x4 block matrix M = [[A, B], [0, D]]
M = np.block([[A, B], [zero_block, D]])
det_M = np.linalg.det(M)
det_formula = np.linalg.det(A) * np.linalg.det(D)

print("Block Matrix M:\n", M)
print(f"det(M) directly: {det_M:.4f}")
print(f"det(A)*det(D):   {det_formula:.4f}")

# 2. LU Decomposition from Scratch (Doolittle Algorithm)
def lu_factorize(mat):
    n = mat.shape[0]
    L = np.eye(n)
    U = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            U[i, j] = mat[i, j] - sum(L[i, k] * U[k, j] for k in range(i))
        for j in range(i + 1, n):
            L[j, i] = (mat[j, i] - sum(L[j, k] * U[k, i] for k in range(i))) / U[i, i]
    return L, U

# Solve Ax = b
A_sys = np.array([[2.0, 1.0, 1.0],
                  [4.0, 1.0, 0.0],
                  [-2.0, 2.0, 1.0]])
b = np.array([4.0, 5.0, 1.0])

L_mat, U_mat = lu_factorize(A_sys)
print("\n--- LU Factorization ---")
print("L:\n", L_mat)
print("U:\n", U_mat)
print("Verification norm ||A - LU||:", np.linalg.norm(A_sys - L_mat @ U_mat))

# Forward substitution: Ly = b
y = np.zeros_like(b)
for i in range(len(b)):
    y[i] = b[i] - sum(L_mat[i, k] * y[k] for k in range(i))

# Back substitution: Ux = y
x = np.zeros_like(b)
for i in range(len(b) - 1, -1, -1):
    x[i] = (y[i] - sum(U_mat[i, k] * x[k] for k in range(i + 1, len(b)))) / U_mat[i, i]

print(f"Solution x = {x}")
print(f"Verification Ax = {A_sys @ x} (expected {b})")

## GATE-Style Practice

**NAT:** Let $M = \begin{bmatrix} A & B \\ 0 & D \end{bmatrix}$ be a $4 \times 4$ block
upper triangular matrix where $A = \begin{bmatrix} 2 & 1 \\ 1 & 3 \end{bmatrix}$ and
$D = \begin{bmatrix} 4 & 2 \\ 1 & 2 \end{bmatrix}$. If $B$ is any arbitrary $2 \times 2$ matrix,
find the determinant $\det(M)$.

**MCQ:** In the Doolittle LU decomposition $A = LU$ (where $L$ has unit diagonal) of
$A = \begin{bmatrix} 2 & 1 \\ 6 & 8 \end{bmatrix}$, what is the value of entry $u_{22}$ in $U$?

A. 3
B. 5
C. 8
D. 2

**MSQ:** Which of the following statements regarding LU decomposition and matrix properties are TRUE?

A. For any non-singular square matrix $A$, there exists a permutation matrix $P$ such that $PA = LU$.
B. If all leading principal submatrices of $A$ are non-singular, then $A$ has a unique decomposition $A = LU$ where $L$ is unit lower triangular.
C. Inverting a triangular matrix takes $O(n^3)$ operations.
D. The determinant of $A = LU$ equals the product of the diagonal elements of $U$.

## Solutions

NAT: **30**.
For a block upper triangular matrix with zero bottom-left block:
$$\det(M) = \det(A) \det(D).$$
$$\det(A) = 2(3) - 1(1) = 5.$$
$$\det(D) = 4(2) - 2(1) = 6.$$
$$\det(M) = 5 \times 6 = 30.$$

MCQ: **B**.
- First row of $U$: $u_{11} = a_{11} = 2$, $u_{12} = a_{12} = 1$.
- First column of $L$: $l_{21} = a_{21} / u_{11} = 6 / 2 = 3$.
- Second row of $U$: $u_{22} = a_{22} - l_{21} u_{12} = 8 - (3)(1) = 5$.

MSQ: **A, B, D**.
- A is true: Gaussian elimination with partial pivoting always yields $PA = LU$.
- B is true: Non-zero pivots ensure no division by zero, yielding a unique Doolittle factorization.
- C is false: Forward/back substitution for triangular matrices takes $O(n^2)$ operations.
- D is true: $\det(A) = \det(L)\det(U) = 1 \times \prod_{i=1}^n u_{ii}$.